In [1]:
from dotenv import load_dotenv
import os

load_dotenv()
print(os.environ.get('OPENAI_API_KEY')[:20])

sk-proj-_fMUdLs8U_td


In [2]:
# import v1.0
from langchain_openai.chat_models import ChatOpenAI
from langchain_core.prompts.prompt import PromptTemplate
from langchain_core.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

# LLM 모델
# FewShotPromptTemplate
from langchain_core.prompts.few_shot import FewShotPromptTemplate

## FewShotPromptTemplate 설계

In [5]:
examples = [
    {
        "country": "프랑스에 대해서 어떻게 알고 있나요?",
        # 원하는 형식의 답변
        "answer": """
            저는 이렇게 알고 있어요.
            수도: 파리
            언어: 프랑스어
            음식: 와인과 치즈
            통화: 유로
        """
    },
    {
        "country": "일본에 대해서 어떻게 알고 있나요?",
        "answer": """
            저는 이렇게 알고 있어요.
            수도: 도쿄
            언어: 일본어
            음식: 초밥과 라멘
            통화: 엔
        """
    },
    {
        "country": "이탈리아에 대해서 어떻게 알고 있나요?",
        "answer": """
            저는 이렇게 알고 있어요.
            수도: 로마
            언어: 이탈리아어
            음식: 피자와 파스타
            통화: 유로
        """
    },
    {
        "country": "캐나다에 대해서 어떻게 알고 있나요?",
        "answer": """
            저는 이렇게 알고 있어요.
            수도: 오타와
            언어: 영어와 프랑스어
            음식: 푸틴과 메이플 시럽
            통화: 캐나다 달러
        """
    }
]

## 2단계: 예시용 프롬프트 템플릿 정의 

In [7]:
example_template = """
    Human: {country}
    AI: {answer}
"""

example_prompt = PromptTemplate.from_template(example_template)

example_prompt

PromptTemplate(input_variables=['answer', 'country'], input_types={}, partial_variables={}, template='\n    Human: {country}\n    AI: {answer}\n')

## 3단계: FewShotPromptTemplate 생성 후 결합

In [13]:
prompt = FewShotPromptTemplate(
    # 질문
    example_prompt=example_prompt,
    # 예시
    examples=examples,
    # 사용자의 질문
    suffix="Human: {country}에 대해서 어떻게 알고 있어요?",
    input_variables=["country"]
)

In [14]:
chat = ChatOpenAI(temperature=0)

In [15]:
chain = prompt | chat

In [17]:
result = chain.invoke(
    {
        "country": "한국"
    }
)

In [18]:
print(result.content)

AI: 
            저는 이렇게 알고 있어요.
            수도: 서울
            언어: 한국어
            음식: 김치와 불고기
            통화: 대한민국 원


In [19]:
result = chain.invoke(
    {
        "country": "영국"
    }
)

In [20]:
print(result.content)

AI: 
            저는 이렇게 알고 있어요.
            수도: 런던
            언어: 영어
            음식: 피쉬 앤 칩스
            통화: 파운드


## FewShotChatMessagePromptTemplate 설계

In [35]:
# v1.0
# ChatModel
from langchain_core.prompts.few_shot import FewShotChatMessagePromptTemplate
from langchain_core.prompts.chat import ChatPromptTemplate

In [36]:
examples = [
    {
        "country": "프랑스에 대해서 어떻게 알고 있나요?",
        # 원하는 형식의 답변
        "answer": """
            저는 이렇게 알고 있어요.
            수도: 파리
            언어: 프랑스어
            음식: 와인과 치즈
            통화: 유로
        """
    },
    {
        "country": "일본에 대해서 어떻게 알고 있나요?",
        "answer": """
            저는 이렇게 알고 있어요.
            수도: 도쿄
            언어: 일본어
            음식: 초밥과 라멘
            통화: 엔
        """
    },
    {
        "country": "이탈리아에 대해서 어떻게 알고 있나요?",
        "answer": """
            저는 이렇게 알고 있어요.
            수도: 로마
            언어: 이탈리아어
            음식: 피자와 파스타
            통화: 유로
        """
    },
    {
        "country": "캐나다에 대해서 어떻게 알고 있나요?",
        "answer": """
            저는 이렇게 알고 있어요.
            수도: 오타와
            언어: 영어와 프랑스어
            음식: 푸틴과 메이플 시럽
            통화: 캐나다 달러
        """
    }
]

## 2단계: 예시용 프롬프트 템플릿 정의 

In [49]:
example = ChatPromptTemplate.from_messages([
    ("human", "{country}에 대해서 어떻게 알고 있어요?"),
    ("ai", "{answer}"),
])

In [50]:
example_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples
)

In [51]:
final = prompt | chain

In [52]:
final_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 지리학 전문가입니다. 짧은 답변을 제공하세요."),
    example_prompt,
    ("human", "{country}에 대해서 어떻게 알고 있어요?")
])

In [53]:
print(final_prompt)

input_variables=['country'] input_types={} partial_variables={} messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='당신은 지리학 전문가입니다. 짧은 답변을 제공하세요.'), additional_kwargs={}), FewShotChatMessagePromptTemplate(examples=[{'country': '프랑스에 대해서 어떻게 알고 있나요?', 'answer': '\n            저는 이렇게 알고 있어요.\n            수도: 파리\n            언어: 프랑스어\n            음식: 와인과 치즈\n            통화: 유로\n        '}, {'country': '일본에 대해서 어떻게 알고 있나요?', 'answer': '\n            저는 이렇게 알고 있어요.\n            수도: 도쿄\n            언어: 일본어\n            음식: 초밥과 라멘\n            통화: 엔\n        '}, {'country': '이탈리아에 대해서 어떻게 알고 있나요?', 'answer': '\n            저는 이렇게 알고 있어요.\n            수도: 로마\n            언어: 이탈리아어\n            음식: 피자와 파스타\n            통화: 유로\n        '}, {'country': '캐나다에 대해서 어떻게 알고 있나요?', 'answer': '\n            저는 이렇게 알고 있어요.\n            수도: 오타와\n            언어: 영어와 프랑스어\n            음식: 푸틴과 메이플 시럽\n            통화: 캐나다 달러\n        

In [54]:
final_chain = final_prompt | chat

In [55]:
result = final_chain.invoke({
    "country": "일본"
})

In [56]:
print(result.content)

일본은 아시아 대륙 동쪽에 위치한 섬나라로, 수도는 도쿄이며 일본어가 공용어로 사용됩니다. 일본은 고대의 전통과 현대의 혁신이 조화로운 나라로, 일본의 문화는 차이나와 함께 동아시아 문화의 중심지 중 하나입니다. 일본은 세계적으로 유명한 음식인 초밥, 라멘, 우동 등 다양한 요리를 자랑하며, 일본의 자연경관과 역사적인 유적지도 많은 관광객들의 관심을 끌고 있습니다. 일본은 또한 선진 기술과 혁신적인 산업으로 유명하며, 일본의 엔이 통화 단위로 사용됩니다.


## LengthBasedExampleSelector 설계

In [57]:
# v1.0
from langchain_core.example_selectors.length_based import LengthBasedExampleSelector

In [58]:
examples = [
    {
        "country": "프랑스에 대해서 어떻게 알고 있나요?",
        # 원하는 형식의 답변
        "answer": """
            저는 이렇게 알고 있어요.
            수도: 파리
            언어: 프랑스어
            음식: 와인과 치즈
            통화: 유로
        """
    },
    {
        "country": "일본에 대해서 어떻게 알고 있나요?",
        "answer": """
            저는 이렇게 알고 있어요.
            수도: 도쿄
            언어: 일본어
            음식: 초밥과 라멘
            통화: 엔
        """
    },
    {
        "country": "이탈리아에 대해서 어떻게 알고 있나요?",
        "answer": """
            저는 이렇게 알고 있어요.
            수도: 로마
            언어: 이탈리아어
            음식: 피자와 파스타
            통화: 유로
        """
    },
    {
        "country": "캐나다에 대해서 어떻게 알고 있나요?",
        "answer": """
            저는 이렇게 알고 있어요.
            수도: 오타와
            언어: 영어와 프랑스어
            음식: 푸틴과 메이플 시럽
            통화: 캐나다 달러
        """
    }
]

In [59]:
example_prompt = PromptTemplate.from_template("Human: {country}\nAI: {answer}")

## Selector 연결

In [64]:
example_selector = LengthBasedExampleSelector(
    examples=examples,
    example_prompt=example_prompt,
    # 예제의 양 허용 (토큰 개수)
    max_length=200
)

In [65]:
prompt = FewShotPromptTemplate(
    example_prompt=example_prompt,
    example_selector=example_selector,
    suffix="Human: {country}에 대해서 어떻게 알고 있나요?",
    input_variables=["country"]
)

In [66]:
prompt.format(**{
    "country": "브라질"
})

'Human: 프랑스에 대해서 어떻게 알고 있나요?\nAI: \n            저는 이렇게 알고 있어요.\n            수도: 파리\n            언어: 프랑스어\n            음식: 와인과 치즈\n            통화: 유로\n        \n\nHuman: 일본에 대해서 어떻게 알고 있나요?\nAI: \n            저는 이렇게 알고 있어요.\n            수도: 도쿄\n            언어: 일본어\n            음식: 초밥과 라멘\n            통화: 엔\n        \n\nHuman: 브라질에 대해서 어떻게 알고 있나요?'

In [67]:
final_chain = prompt | chat

In [68]:
result = final_chain.invoke({
    "country": "독일"
})

In [69]:
print(result.content)

AI: 
            저는 이렇게 알고 있어요.
            수도: 베를린
            언어: 독일어
            음식: 소세지와 맥주
            통화: 유로


## Chat 모델(LengthBasedExampleSelector) 

In [70]:
examples = [
    {
        "country": "프랑스에 대해서 어떻게 알고 있나요?",
        # 원하는 형식의 답변
        "answer": """
            저는 이렇게 알고 있어요.
            수도: 파리
            언어: 프랑스어
            음식: 와인과 치즈
            통화: 유로
        """
    },
    {
        "country": "일본에 대해서 어떻게 알고 있나요?",
        "answer": """
            저는 이렇게 알고 있어요.
            수도: 도쿄
            언어: 일본어
            음식: 초밥과 라멘
            통화: 엔
        """
    },
    {
        "country": "이탈리아에 대해서 어떻게 알고 있나요?",
        "answer": """
            저는 이렇게 알고 있어요.
            수도: 로마
            언어: 이탈리아어
            음식: 피자와 파스타
            통화: 유로
        """
    },
    {
        "country": "캐나다에 대해서 어떻게 알고 있나요?",
        "answer": """
            저는 이렇게 알고 있어요.
            수도: 오타와
            언어: 영어와 프랑스어
            음식: 푸틴과 메이플 시럽
            통화: 캐나다 달러
        """
    }
]

In [72]:
length_prompt = PromptTemplate(
    input_variables=["country", "answer"],
    template="""
        Human: {country}에 대해서 어떻게 알고 있어요? \n
        AI: {answer}
    """
)

example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{country}에 대해서 어떻게 알고 있어요?"),
    ("ai", "{answer}")
])

example_selector = LengthBasedExampleSelector(
    examples=examples,
    example_prompt=length_prompt,
    max_length=200
)

fewshot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples
)

In [74]:
final_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 지리학 전문가입니다."),
    fewshot_prompt,
    ("human", "{country}에 대해 어떻게 알고 있어요?"),
])

In [75]:
chain = final_prompt | chat

In [76]:
result = chain.invoke({
    "country": "대한민국"
})

In [77]:
print(result.content)

대한민국에 대해 알고 있는 정보는 다음과 같습니다.
            수도: 서울
            언어: 한국어
            음식: 김치, 불고기, 비빔밥 등
            통화: 대한민국 원 (KRW)
            관광지: 경복궁, 남산타워, 부산 해운대 등
            문화: 한복, 태권도, 한류 등
            경제: 세계적인 IT 기업들과 자동차 제조사들이 발달한 고도의 경제력을 가지고 있습니다.
